# 14 — Color + TabNet | Original & Masked
Gereksinim: `pip install pytorch-tabnet torch`. Eğitimde yalnızca train, early stopping için **train içinden ayrılan** internal holdout kullanılır. Validation yalnızca kıyaslama içindir. Test kullanılmaz. Grid yerine sınırlı random parametre araması.

In [1]:
from pathlib import Path
import sys, json, time
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(ROOT) not in sys.path: sys.path.insert(0, str(ROOT))
from src.config import BASE_DIR
from src.data_loader import load_dataset
FEATURE_DIR = BASE_DIR / "outputs" / "features"
RESULT_DIR = BASE_DIR / "outputs" / "models" / "deep_learning"
RESULT_DIR.mkdir(parents=True, exist_ok=True)
df = load_dataset(require_masks=True).reset_index(drop=True)
y = df['label'].to_numpy(dtype=str)
ids = df['image'].to_numpy(dtype=str)
classes = np.unique(y)
idx = np.arange(len(y))
dev_idx, test_idx = train_test_split(idx, test_size=.15, random_state=42, stratify=y)
train_idx, val_idx = train_test_split(dev_idx, test_size=.15/.85, random_state=42, stratify=y[dev_idx])
assert len(set(train_idx)&set(val_idx)) == len(set(train_idx)&set(test_idx)) == len(set(val_idx)&set(test_idx)) == 0
print('Train:',len(train_idx),'Val:',len(val_idx),'Test (kullanılmayacak):',len(test_idx))
# ÖNEMLİ: Aynı lezyondan farklı görüntüler varsa lesion_id ile group split ayrıca uygulanmalı.


Train: 7009 Val: 1503 Test (kullanılmayacak): 1503


In [2]:
paths = {'original': FEATURE_DIR/'X_original_color_300x225.npy', 'masked':FEATURE_DIR/'X_masked_color_300x225.npy'}
ids_file = FEATURE_DIR/'image_ids_color_300x225.npy'
y_file = FEATURE_DIR/'y_labels_color_300x225.npy'
assert np.array_equal(np.load(ids_file,allow_pickle=False).astype(str), ids)
assert np.array_equal(np.load(y_file,allow_pickle=False).astype(str), y)
for p in paths.values():
    x0=np.load(p,mmap_mode='r',allow_pickle=False)
    assert x0.shape == (len(y),439), (p,x0.shape)
print('Color ID/etiket sırası doğrulandı.')


Color ID/etiket sırası doğrulandı.


In [4]:
%pip install pytorch-tabnet torch


Note: you may need to restart the kernel to use updated packages.


In [6]:
# Bir kez: %pip install pytorch-tabnet torch
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.preprocessing import LabelEncoder,StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score
import torch
encoder=LabelEncoder().fit(y)
y_int=encoder.transform(y)
PARAMS=[{'n_d':8,'n_a':8,'n_steps':3,'gamma':1.3,'lambda_sparse':1e-4},
        {'n_d':16,'n_a':16,'n_steps':3,'gamma':1.5,'lambda_sparse':1e-4},
        {'n_d':16,'n_a':16,'n_steps':4,'gamma':1.3,'lambda_sparse':1e-5}]
EPOCHS=90;BATCH_SIZE=256
print('PyTorch CUDA:',torch.cuda.is_available())

PyTorch CUDA: False


In [7]:
def store_result(model_name,variant,pred,train_pred,best_params,cv_score=None,seconds=None):
    prefix=RESULT_DIR/f'{model_name}_{variant}'
    r={'Model':model_name,'Variant':variant,'CV_Macro_F1':cv_score,
       'Train_Macro_F1':float(f1_score(y[train_idx],train_pred,average='macro',zero_division=0)),
       'Val_Accuracy':float(accuracy_score(y[val_idx],pred)),
       'Val_Macro_F1':float(f1_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Precision':float(precision_score(y[val_idx],pred,average='macro',zero_division=0)),
       'Val_Macro_Recall':float(recall_score(y[val_idx],pred,average='macro',zero_division=0)),
       'MEL_Recall':float(recall_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'MEL_Precision':float(precision_score(y[val_idx],pred,labels=['MEL'],average='macro',zero_division=0)),
       'Predicted_Classes':int(len(np.unique(pred))),'Best_Params':json.dumps(best_params,ensure_ascii=False,default=str),
       'Search_Seconds':seconds,'Train_N':len(train_idx),'Val_N':len(val_idx),'Test_N':len(test_idx)}
    pd.DataFrame({'image_id':ids[val_idx], 'actual':y[val_idx], 'predicted':pred}).to_csv(str(prefix)+'_val_predictions.csv',index=False)
    pd.DataFrame(confusion_matrix(y[val_idx],pred,labels=classes),index=classes,columns=classes).to_csv(str(prefix)+'_confusion_matrix.csv')
    print(classification_report(y[val_idx],pred,labels=classes,zero_division=0))
    return r


In [8]:
rows=[]
for variant,path in paths.items():
    started=time.time();X=np.load(path,mmap_mode='r',allow_pickle=False)
    Xt=np.asarray(X[train_idx],dtype=np.float32);Xv=np.asarray(X[val_idx],dtype=np.float32)
    # TabNet ağının optimizasyonunu hızlandırmak için ölçekleme. Yalnız train fit.
    s=StandardScaler().fit(Xt);Xt=s.transform(Xt).astype('float32');Xv=s.transform(Xv).astype('float32')
    # İç validation sadece train içinden; dış validation hiperparametre seçimine girmez.
    a,b=train_test_split(np.arange(len(train_idx)),test_size=.15,stratify=y[train_idx],random_state=43)
    best_score=-1;best_model=None;best_params=None;logs=[]
    for params in PARAMS:
        model=TabNetClassifier(**params,seed=42,verbose=0,device_name='auto')
        model.fit(Xt[a],y_int[train_idx][a],eval_set=[(Xt[b],y_int[train_idx][b])],
            eval_name=['inner_val'],eval_metric=['logloss'],max_epochs=EPOCHS,patience=12,
            batch_size=BATCH_SIZE,virtual_batch_size=64,num_workers=0,drop_last=False)
        inner=model.predict(Xt[b]);score=f1_score(y_int[train_idx][b],inner,average='macro',zero_division=0)
        logs.append({**params,'Inner_Val_Macro_F1':score})
        print(variant,params,'inner Macro F1:',score)
        if score>best_score:best_score=score;best_model=model;best_params=params
    pred=encoder.inverse_transform(best_model.predict(Xv).astype(int))
    tr=encoder.inverse_transform(best_model.predict(Xt).astype(int))
    prefix=RESULT_DIR/f'TabNet_Color_{variant}'
    pd.DataFrame(logs).to_csv(str(prefix)+'_search_results.csv',index=False)
    import joblib
    joblib.dump({'scaler':s,'encoder':encoder},str(prefix)+'_preprocessors.joblib')
    best_model.save_model(str(prefix)+'_model')
    r=store_result('TabNet_Color',variant,pred,tr,best_params,round(best_score,5),round(time.time()-started,1));rows.append(r)
summary=pd.DataFrame(rows).sort_values('Val_Macro_F1',ascending=False)
summary.to_csv(RESULT_DIR/'TabNet_Color_summary.csv',index=False);display(summary)



Early stopping occurred at epoch 55 with best_epoch = 43 and best_inner_val_logloss = 0.78552


c:\Users\Deniz TOZAN\miniconda3\envs\dl_env\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


original {'n_d': 8, 'n_a': 8, 'n_steps': 3, 'gamma': 1.3, 'lambda_sparse': 0.0001} inner Macro F1: 0.3053569912094252

Early stopping occurred at epoch 30 with best_epoch = 18 and best_inner_val_logloss = 0.8108


c:\Users\Deniz TOZAN\miniconda3\envs\dl_env\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


original {'n_d': 16, 'n_a': 16, 'n_steps': 3, 'gamma': 1.5, 'lambda_sparse': 0.0001} inner Macro F1: 0.2645520325843819

Early stopping occurred at epoch 54 with best_epoch = 42 and best_inner_val_logloss = 0.83472


...

c:\Users\Deniz TOZAN\miniconda3\envs\dl_env\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


masked {'n_d': 16, 'n_a': 16, 'n_steps': 3, 'gamma': 1.5, 'lambda_sparse': 0.0001} inner Macro F1: 0.36838293943916256
Stop training because you reached max_epochs = 90 with best_epoch = 78 and best_inner_val_logloss = 0.75326


c:\Users\Deniz TOZAN\miniconda3\envs\dl_env\Lib\site-packages\pytorch_tabnet\callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


masked {'n_d': 16, 'n_a': 16, 'n_steps': 4, 'gamma': 1.3, 'lambda_sparse': 1e-05} inner Macro F1: 0.40583816299281084
Successfully saved model at C:\Users\Deniz TOZAN\OneDrive - Analythinx Bilişim Hizmetleri Ticaret Anonim Şirketi\WORK\Calismalar\uygulama\Kodlar\Health\outputs\models\deep_learning\TabNet_Color_masked_model.zip
              precision    recall  f1-score   support

       AKIEC       0.33      0.45      0.38        49
         BCC       0.49      0.48      0.48        77
         BKL       0.45      0.36      0.40       165
          DF       0.00      0.00      0.00        18
         MEL       0.42      0.26      0.32       167
          NV       0.82      0.90      0.86      1006
        VASC       0.52      0.57      0.55        21

    accuracy                           0.72      1503
   macro avg       0.43      0.43      0.43      1503
weighted avg       0.69      0.72      0.70      1503



,Model,Variant,CV_Macro_F1,Train_Macro_F1,Val_Accuracy,Val_Macro_F1,Val_Macro_Precision,Val_Macro_Recall,MEL_Recall,MEL_Precision,Predicted_Classes,Best_Params,Search_Seconds,Train_N,Val_N,Test_N
1,TabNet_Color,masked,0.45094,0.496340,0.715902,0.426513,0.433008,0.430515,0.257485,0.421569,6,"{""n_d"": 8, ""n_a"": 8, ""n_steps"": 3, ""gamma"": 1....",330.8,7009,1503,1503
0,TabNet_Color,original,0.30536,0.324748,0.703260,0.290004,0.340198,0.300086,0.149701,0.462963,5,"{""n_d"": 8, ""n_a"": 8, ""n_steps"": 3, ""gamma"": 1....",222.0,7009,1503,1503
